In [ ]:
!pip install -q transformers torch accelerate

import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_adi = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_adi)

sinif_isimleri = ["Haberleşme", "Motor/İtki", "Aviyonik"]

# Eğitim Verisi (12 cümle)
egitim_verisi = [
    ("C-Bant veri linki bağlantısında sinyal zayıflaması yaşandı.", 0),
    ("Yer kontrol istasyonu ile telemetri iletişimi kesildi.", 0),
    ("Anten yönlendirme modülü frekans kilidini kaybetti.", 0),
    ("Uydu haberleşme terminalinde paket kaybı arttı.", 0),
    ("Turbofan motor egzoz gazı sıcaklığı kritik eşiği aştı.", 1),
    ("Yakıt pompası basıncı düştüğü için itki gücü azaldı.", 1),
    ("Türbin devri dalgalanıyor ve yakıt akışı düzensiz.", 1),
    ("Motor yağ sıcaklığı yüksek uyarısı aktifleşti.", 1),
    ("Lazer altimetre ve pitot tüpü irtifa verisi uyuşmuyor.", 2),
    ("Uçuş kontrol bilgisayarı yedek IMU sensörüne geçti.", 2),
    ("GPS alıcısı uydu sinyallerini kaybetti, INS moduna geçildi.", 2),
    ("Elektro-optik kamera gimbal stabilizasyon hatası verdi.", 2),
]

# Test Verisi (Eğitimde hiç görülmeyen 6 cümle)
test_verisi = [
    ("Datalink anteni sinyal kaybetti ve iletişim koptu.", 0),
    ("Yer istasyonu ile veri paketi aktarımı durdu.", 0),
    ("Motor yakıt akışında düşüş ve türbin ısınması var.", 1),
    ("İtki sistemi beklenenden düşük güç üretiyor.", 1),
    ("GPS ve INS navigasyon sensörleri arasında sapma oluştu.", 2),
    ("Altimetre sensörü hatalı irtifa değeri gönderiyor.", 2),
]

egitim_x = tokenizer([x[0] for x in egitim_verisi], padding=True, truncation=True, return_tensors="pt").to(device)
egitim_y = torch.tensor([x[1] for x in egitim_verisi]).to(device)
test_x = tokenizer([x[0] for x in test_verisi], padding=True, truncation=True, return_tensors="pt").to(device)
test_y = torch.tensor([x[1] for x in test_verisi]).to(device)

def dogruluk_hesapla(aktif_model):
    aktif_model.eval()
    with torch.no_grad():
        tahminler = aktif_model(**test_x).logits.argmax(dim=-1)
    return ((tahminler == test_y).sum().item() / len(test_y)) * 100

print(f"Hazırlık tamamlandı. Cihaz: {device} | Eğitim: {len(egitim_verisi)} | Test: {len(test_verisi)}")

Hazırlık tamamlandı. Cihaz: cuda | Eğitim: 12 | Test: 6


In [ ]:
print("=" * 75)
print("DENEY 1: PRUNING (L1 UNSTRUCTURED BUDAMA - %30 SEYREKLİK)")
print("=" * 75)

# 1. Sıfırdan bağımsız model yükleme ve eğitim
pruning_model = AutoModelForSequenceClassification.from_pretrained(model_adi, num_labels=3).to(device)
p_opt = torch.optim.AdamW(pruning_model.parameters(), lr=5e-5)

pruning_model.train()
for _ in range(20):
    p_opt.zero_grad()
    pruning_model(**egitim_x, labels=egitim_y).loss.backward()
    p_opt.step()

# 2. Budama öncesi ölçüm
toplam_param = sum(p.numel() for p in pruning_model.parameters())
aktif_once = sum(torch.count_nonzero(p).item() for p in pruning_model.parameters())
acc_once = dogruluk_hesapla(pruning_model)

# 3. Linear katmanlardaki ağırlıkların %30'unu kalıcı olarak sıfırlama
budanan_katman = 0
for modul in pruning_model.modules():
    if isinstance(modul, nn.Linear):
        prune.l1_unstructured(modul, name="weight", amount=0.30)
        prune.remove(modul, "weight")
        budanan_katman += 1

# 4. Budama sonrası ölçüm
aktif_sonra = sum(torch.count_nonzero(p).item() for p in pruning_model.parameters())
acc_sonra = dogruluk_hesapla(pruning_model)

print(f"Budanan Linear Katman Sayısı : {budanan_katman}")
print(f"Toplam Parametre             : {toplam_param:,}")
print(f"Budama Öncesi Aktif Parametre: {aktif_once:,} (%100.0) | Test Doğruluğu: %{acc_once:.1f}")
print(f"Budama Sonrası Aktif Parametre: {aktif_sonra:,} (%{aktif_sonra/toplam_param*100:.1f}) | Test Doğruluğu: %{acc_sonra:.1f}")
print(f"Sıfırlanan Bağlantı Sayısı   : {aktif_once - aktif_sonra:,}")

# Bellek temizliği (Bir sonraki deney sıfırdan başlasın)
del pruning_model
torch.cuda.empty_cache()

DENEY 1: PRUNING (L1 UNSTRUCTURED BUDAMA - %30 SEYREKLİK)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Budanan Linear Katman Sayısı : 74
Toplam Parametre             : 110,619,651
Budama Öncesi Aktif Parametre: 110,619,651 (%100.0) | Test Doğruluğu: %83.3
Budama Sonrası Aktif Parametre: 84,961,621 (%76.8) | Test Doğruluğu: %50.0
Sıfırlanan Bağlantı Sayısı   : 25,658,030


In [ ]:
print("=" * 75)
print("DENEY 2: KNOWLEDGE DISTILLATION (HASARSIZ 12 KATMAN ÖĞRETMEN -> 2 KATMAN ÖĞRENCİ)")
print("=" * 75)

# 1. Sıfırdan hasarsız Öğretmen Model (12 Katman) yükleme ve eğitim
teacher_model = AutoModelForSequenceClassification.from_pretrained(model_adi, num_labels=3).to(device)
t_opt = torch.optim.AdamW(teacher_model.parameters(), lr=5e-5)

teacher_model.train()
for _ in range(20):
    t_opt.zero_grad()
    teacher_model(**egitim_x, labels=egitim_y).loss.backward()
    t_opt.step()

# 2. Sıfırdan Öğrenci Model (2 Katman) kurulumu
student_config = AutoConfig.from_pretrained(model_adi, num_labels=3, num_hidden_layers=2)
student_model = AutoModelForSequenceClassification.from_config(student_config).to(device)
student_model.bert.embeddings.load_state_dict(teacher_model.bert.embeddings.state_dict())

# 3. Knowledge Distillation Eğitimi (Soft Logits + Temperature T + Hard Labels)
T = 4.0
alpha = 0.7
s_opt = torch.optim.AdamW(student_model.parameters(), lr=2e-4)

teacher_model.eval()
with torch.no_grad():
    soft_teacher = F.softmax(teacher_model(**egitim_x).logits / T, dim=-1)

student_model.train()
for _ in range(45):
    s_opt.zero_grad()
    s_logits = student_model(**egitim_x).logits
    distill_loss = nn.KLDivLoss(reduction="batchmean")(F.log_softmax(s_logits / T, dim=-1), soft_teacher) * (T ** 2)
    student_loss = nn.CrossEntropyLoss()(s_logits, egitim_y)
    (alpha * distill_loss + (1 - alpha) * student_loss).backward()
    s_opt.step()

# 4. Performans, Hız ve Olasılık Dağılımı Karşılaştırması
def model_boyutu_mb(m):
    return sum(p.nelement() * p.element_size() for p in m.parameters()) / (1024 ** 2)

def hiz_olc_ms(m, girdi, tekrar=100):
    m.eval()
    with torch.no_grad():
        for _ in range(10):
            _ = m(**girdi)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(tekrar):
            _ = m(**girdi)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.time()
    return ((t1 - t0) / tekrar) * 1000

tek_ornek = tokenizer("Datalink anteni sinyal kaybetti ve iletişim koptu.", return_tensors="pt").to(device)

t_param = sum(p.numel() for p in teacher_model.parameters())
s_param = sum(p.numel() for p in student_model.parameters())
t_mb, s_mb = model_boyutu_mb(teacher_model), model_boyutu_mb(student_model)
t_ms, s_ms = hiz_olc_ms(teacher_model, tek_ornek), hiz_olc_ms(student_model, tek_ornek)
t_acc, s_acc = dogruluk_hesapla(teacher_model), dogruluk_hesapla(student_model)

print(f"{'METRİK':<28} | {'ÖĞRETMEN (12 Katman)':<22} | {'ÖĞRENCİ (2 Katman)':<22}")
print("-" * 78)
print(f"{'Toplam Parametre':<28} | {t_param:<22,} | {s_param:<22,}")
print(f"{'Bellek Boyutu (MB)':<28} | {t_mb:<22.1f} | {s_mb:<22.1f}")
print(f"{'Ortalama Hız (ms/sorgu)':<28} | {t_ms:<22.2f} | {s_ms:<22.2f}")
print(f"{'Test Doğruluğu (Accuracy)':<28} | %{t_acc:<21.1f} | %{s_acc:<21.1f}")
print("-" * 78)

# Test cümlelerindeki olasılık dağılımlarını (Soft Probabilities) yan yana görme
teacher_model.eval()
student_model.eval()
with torch.no_grad():
    t_probs = F.softmax(teacher_model(**test_x).logits, dim=-1).cpu().numpy() * 100
    s_probs = F.softmax(student_model(**test_x).logits, dim=-1).cpu().numpy() * 100

print("\nTEST CÜMLELERİNDE OLASILIK DAĞILIMI KARŞILAŞTIRMASI:")
for i, (cumle, gercek_id) in enumerate(test_verisi[:3]):
    print(f"\nTest {i+1}: '{cumle}' (Gerçek: {sinif_isimleri[gercek_id]})")
    print(f"  -> Öğretmen : Haberleşme %{t_probs[i][0]:4.1f} | Motor %{t_probs[i][1]:4.1f} | Aviyonik %{t_probs[i][2]:4.1f}")
    print(f"  -> Öğrenci  : Haberleşme %{s_probs[i][0]:4.1f} | Motor %{s_probs[i][1]:4.1f} | Aviyonik %{s_probs[i][2]:4.1f}")

DENEY 2: KNOWLEDGE DISTILLATION (HASARSIZ 12 KATMAN ÖĞRETMEN -> 2 KATMAN ÖĞRENCİ)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


METRİK                       | ÖĞRETMEN (12 Katman)   | ÖĞRENCİ (2 Katman)    
------------------------------------------------------------------------------
Toplam Parametre             | 110,619,651            | 39,740,931            
Bellek Boyutu (MB)           | 422.0                  | 151.6                 
Ortalama Hız (ms/sorgu)      | 12.15                  | 2.79                  
Test Doğruluğu (Accuracy)    | %66.7                  | %100.0                
------------------------------------------------------------------------------

TEST CÜMLELERİNDE OLASILIK DAĞILIMI KARŞILAŞTIRMASI:

Test 1: 'Datalink anteni sinyal kaybetti ve iletişim koptu.' (Gerçek: Haberleşme)
  -> Öğretmen : Haberleşme %94.5 | Motor % 2.0 | Aviyonik % 3.6
  -> Öğrenci  : Haberleşme %80.5 | Motor % 9.8 | Aviyonik % 9.6

Test 2: 'Yer istasyonu ile veri paketi aktarımı durdu.' (Gerçek: Haberleşme)
  -> Öğretmen : Haberleşme %91.6 | Motor % 6.4 | Aviyonik % 2.0
  -> Öğrenci  : Haberleşme %87.3 | Motor